# Phase 1 — Delta Lake Fundamentals

**Concept.** Delta Lake is an open-source storage layer that sits on top of
Parquet and adds ACID transactions, a transaction log, time travel, and
schema evolution to a plain data lake. Every write (`INSERT`/`UPDATE`/
`DELETE`/`MERGE`) is recorded as a JSON commit in `_delta_log/`, which is
what gives you atomicity (no partial/corrupt reads mid-write), consistent
snapshots, and the ability to query *any previous version* of a table.

Official docs: [docs.delta.io/latest/delta-intro](https://docs.delta.io/latest/delta-intro.html)
· [Databricks: What is Delta Lake?](https://docs.databricks.com/en/delta/index.html)

**Why it matters professionally.** Plain Parquet has no transaction
guarantees — a failed Spark job mid-write can leave a table half-written,
concurrent writers can corrupt each other's output, and there's no built-in
way to `UPDATE`/`DELETE`/`MERGE` rows or recover a bad write. Delta Lake is
the de-facto standard for lakehouse architectures (Databricks, Azure
Synapse, and increasingly plain open-source Spark) precisely because it
closes these gaps without giving up the cost/scale benefits of a data lake.

**Where this fits in DataForge AI.** We're extending the Phase 0 pipeline
into a **medallion architecture**:

```mermaid
flowchart LR
    A[Raw Parquet<br/>data/raw/] -->|read_trips| B[Bronze<br/>Delta table<br/>typed, unfiltered]
    B -->|clean_trips + join_zones| C[Silver<br/>Delta table<br/>cleaned, enriched]
    C -->|hourly_demand, etc.| D[Gold<br/>Delta table<br/>aggregated, business-level]
```

- **Bronze** = raw data cast to proper types (our existing `read_trips`), no
  filtering — a durable, replayable copy of what we ingested.
- **Silver** = cleaned + zone-enriched (`clean_trips` + `join_zones`) — the
  conformed, query-ready layer.
- **Gold** (later) = aggregated tables for direct consumption (e.g. our
  `hourly_demand` output).

**Caveat:** our base image runs PySpark 3.5.3; we're using `delta-spark`
3.2.1 (the latest release compatible with Spark 3.5.x). APIs shown here are
stable across recent Delta versions, but if you look things up on
docs.delta.io check the compatibility matrix if something looks off.


In [1]:
# Setup: SparkSession configured for Delta Lake.
# configure_spark_with_delta_pip() reads the installed delta-spark version
# and adds the matching Maven coordinate to spark.jars.packages, so Spark's
# Ivy resolver downloads the actual delta-core/delta-storage JARs the JVM
# needs (delta-spark's pip package only ships the Python bindings). First
# run needs internet access and takes a few seconds longer; the JARs are
# cached under ~/.ivy2 for the life of the container after that.
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from dataforge_ai import read_trips, clean_trips, join_zones

builder = (
    SparkSession.builder
    .appName("DataForge-Phase1-DeltaLake")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Effective driver memory:", spark.conf.get("spark.driver.memory"))
print("Delta extension active:", spark.conf.get("spark.sql.extensions"))

RAW = "../data/raw"
DELTA_ROOT = "../data/delta"
BRONZE_PATH = f"{DELTA_ROOT}/bronze/trips"
SILVER_PATH = f"{DELTA_ROOT}/silver/trips_enriched"
assert os.path.exists(RAW), f"Can't find {RAW}. Current dir is: {os.getcwd()}"
os.makedirs(DELTA_ROOT, exist_ok=True)

Effective driver memory: 4g
Delta extension active: io.delta.sql.DeltaSparkSessionExtension


## Small example: write, read, and inspect the transaction log

Before touching the real dataset, let's see the mechanics on a tiny
DataFrame: write it as Delta, read it back, and look at what actually
landed on disk in `_delta_log/`.


In [2]:
demo_path = f"{DELTA_ROOT}/_demo/toy"

toy = spark.createDataFrame(
    [(1, "a"), (2, "b"), (3, "c")], ["id", "letter"]
)
toy.write.format("delta").mode("overwrite").save(demo_path)

readback = spark.read.format("delta").load(demo_path)
readback.show()

print("Files under _delta_log/:")
for f in sorted(os.listdir(f"{demo_path}/_delta_log")):
    print(" ", f)

print("\nCommit 0 contents:")
with open(f"{demo_path}/_delta_log/00000000000000000000.json") as fh:
    print(fh.read())

+---+------+
| id|letter|
+---+------+
|  1|     a|
|  2|     b|
|  3|     c|
+---+------+

Files under _delta_log/:
  .00000000000000000000.json.crc
  00000000000000000000.json
  _commits

Commit 0 contents:
{"commitInfo":{"timestamp":1788271959649,"operation":"WRITE","operationParameters":{"mode":"Overwrite","partitionBy":"[]"},"isolationLevel":"Serializable","isBlindAppend":false,"operationMetrics":{"numFiles":"4","numOutputRows":"3","numOutputBytes":"2474"},"engineInfo":"Apache-Spark/3.5.3 Delta-Lake/3.2.1","txnId":"65dc80d0-a921-4389-a1b1-d0ab7d002350"}}
{"metaData":{"id":"27e93e3a-2b0a-4f79-8b2a-a2ca682c7ac2","format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"letter\",\"type\":\"string\",\"nullable\":true,\"metadata\":{}}]}","partitionColumns":[],"configuration":{},"createdTime":1788271957354}}
{"protocol":{"minReaderVersion":1,"minWriterVersion":2}}
{"add"

## Hands-on: build the Bronze layer

Bronze = raw data cast to proper types, unfiltered — a durable, replayable
copy of what we ingested. This reuses `read_trips` from our `dataforge_ai`
package (no duplicated casting logic). We partition by `pickup_month` since
that's a natural, low-cardinality query filter for time-series ingestion
(and how a real streaming/batch pipeline would land daily/monthly batches).


In [3]:
paths = [
    f"{RAW}/yellow_tripdata_2023-01.parquet",
    f"{RAW}/yellow_tripdata_2023-02.parquet",
    f"{RAW}/yellow_tripdata_2023-03.parquet",
]

trips_raw = read_trips(spark, paths).withColumn(
    "pickup_month", F.date_format("tpep_pickup_datetime", "yyyy-MM")
)

(
    trips_raw.write.format("delta")
    .mode("overwrite")
    .partitionBy("pickup_month")
    .save(BRONZE_PATH)
)

bronze = spark.read.format("delta").load(BRONZE_PATH)
print("Bronze rows:", bronze.count())
bronze.groupBy("pickup_month").count().orderBy("pickup_month").show()

Bronze rows: 9384487
+------------+-------+
|pickup_month|  count|
+------------+-------+
|     2001-01|      3|
|     2002-12|      2|
|     2003-01|      3|
|     2008-12|      8|
|     2009-01|      1|
|     2014-11|      1|
|     2022-10|     11|
|     2022-12|     25|
|     2023-01|3066726|
|     2023-02|2914003|
|     2023-03|3403619|
|     2023-04|     85|
+------------+-------+



## Hands-on: build the Silver layer

Silver = cleaned + zone-enriched — reusing `clean_trips` and `join_zones`
from the package, reading Bronze as the source instead of raw Parquet
(this is the medallion pattern: each layer reads from the previous Delta
table, never straight from raw files again).


In [4]:
zones = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{RAW}/taxi_zone_lookup.csv")
)

bronze = spark.read.format("delta").load(BRONZE_PATH)

trips_silver = (
    join_zones(clean_trips(bronze), zones)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
)

(
    trips_silver.write.format("delta")
    .mode("overwrite")
    .partitionBy("pickup_month")
    .save(SILVER_PATH)
)

silver = spark.read.format("delta").load(SILVER_PATH)
print("Silver rows:", silver.count())
silver.select(
    "pickup_zone", "pickup_borough", "dropoff_zone", "pickup_hour", "fare_amount"
).show(5)

Silver rows: 9301798
+--------------------+--------------+--------------------+-----------+-----------+
|         pickup_zone|pickup_borough|        dropoff_zone|pickup_hour|fare_amount|
+--------------------+--------------+--------------------+-----------+-----------+
|        West Village|     Manhattan|      Newark Airport|          4|      87.02|
|Upper East Side S...|     Manhattan|Upper East Side N...|         12|        5.8|
|Upper West Side S...|     Manhattan|      Newark Airport|          4|       89.5|
|Upper East Side N...|     Manhattan|Upper East Side N...|         12|        5.1|
|         Murray Hill|     Manhattan|      Newark Airport|          4|       77.6|
+--------------------+--------------+--------------------+-----------+-----------+
only showing top 5 rows



## ACID in practice: the transaction log

Every write we just did is a commit. `DESCRIBE HISTORY` reads directly from
`_delta_log/` — this is what gives Delta tables audit trail, and what
`versionAsOf` time travel (next section) relies on.


In [9]:
# Note: the `delta.`<path>`` SQL syntax requires an ABSOLUTE path -- Spark's
# SQL table-resolution doesn't apply the same relative-path handling that
# DataFrameReader.load() does. DeltaTable.forPath() sidesteps this entirely
# by using the same path resolution as .load()/.save(), so relative paths
# work fine here.
from delta.tables import DeltaTable

DeltaTable.forPath(spark, BRONZE_PATH).history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+----------------------------------------------------+
|version|timestamp              |operation|operationParameters                                 |
+-------+-----------------------+---------+----------------------------------------------------+
|0      |2026-09-01 14:42:14.455|WRITE    |{mode -> Overwrite, partitionBy -> ["pickup_month"]}|
+-------+-----------------------+---------+----------------------------------------------------+



## Schema evolution

A common real-world case: a new batch arrives with an extra column your
existing table doesn't have. Plain Parquet either fails or silently drops
it depending on how you read it. Delta lets you evolve the schema
explicitly with `mergeSchema` — here we simulate re-ingesting January with
an added `ingestion_date` column.


In [ ]:
# Simulate a small new batch (3 sample rows) that includes an extra column
# our current Bronze schema doesn't have yet.
new_batch = bronze.limit(3).withColumn("ingestion_date", F.current_date())

print("New batch schema has ingestion_date:", "ingestion_date" in new_batch.columns)

try:
    new_batch.write.format("delta").mode("append").save(BRONZE_PATH)
except Exception as e:
    print("Failed without mergeSchema, as expected:")
    print(type(e).__name__, "-", str(e)[:200])

# Now retry with mergeSchema=true, which evolves the table schema in place.
new_batch.write.format("delta").mode("append").option("mergeSchema", "true").save(BRONZE_PATH)

bronze_v2 = spark.read.format("delta").load(BRONZE_PATH)
bronze_v2.printSchema()
print("Rows where ingestion_date is set:", bronze_v2.filter(F.col("ingestion_date").isNotNull()).count())
print("Rows where ingestion_date is null (pre-existing data):", bronze_v2.filter(F.col("ingestion_date").isNull()).count())

## Time travel

Because every write is a numbered commit, you can read the table exactly
as it existed at any prior version (or timestamp) — useful for debugging a
bad pipeline run, reproducing a report, or auditing what changed.


In [ ]:
history = DeltaTable.forPath(spark, BRONZE_PATH).history().select("version", "operation")
history.orderBy("version").show()

v0_count = spark.read.format("delta").option("versionAsOf", 0).load(BRONZE_PATH).count()
latest_count = spark.read.format("delta").load(BRONZE_PATH).count()

print(f"Version 0 (initial load): {v0_count:,} rows")
print(f"Latest version (after schema-evolved append): {latest_count:,} rows")
print(f"Difference: {latest_count - v0_count} rows (the 3-row demo batch)")

## `MERGE` (upsert)

The other headline Delta feature: atomic upserts. Say a downstream fare
correction arrives for a handful of trips — with plain Parquet you'd have
to rewrite the whole partition; with Delta, `MERGE` updates matching rows
and inserts new ones in a single atomic transaction via `DeltaTable`
([docs.delta.io/latest/delta-update](https://docs.delta.io/latest/delta-update.html#upsert-into-a-table-using-merge)).


In [ ]:
from delta.tables import DeltaTable

# Fields that identify "the same trip" -- deliberately excludes fare_amount,
# since that's the field the correction is changing.
match_keys = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID", "trip_distance",
]

silver = spark.read.format("delta").load(SILVER_PATH)
sample = silver.limit(3).select(*match_keys, "fare_amount")
print("Fare amounts before correction:")
sample.show()

corrections = sample.withColumn("fare_amount", F.col("fare_amount") + 10.0)

silver_table = DeltaTable.forPath(spark, SILVER_PATH)
merge_condition = " AND ".join(f"t.{c} = s.{c}" for c in match_keys)

(
    silver_table.alias("t")
    .merge(corrections.alias("s"), merge_condition)
    .whenMatchedUpdate(set={"fare_amount": "s.fare_amount"})
    .execute()
)

updated = (
    spark.read.format("delta").load(SILVER_PATH)
    .join(corrections.select(*match_keys), match_keys, "inner")
    .select(*match_keys, "fare_amount")
)
print("Fare amounts after correction (+10):")
updated.show()

DeltaTable.forPath(spark, SILVER_PATH).history().select(
    "version", "operation", "operationMetrics"
).orderBy("version").show(truncate=False)

## Challenge 1: build the Gold layer

Gold = aggregated, business-level tables built from Silver — reusing
`hourly_demand` from notebook 04 (already in the package), no new logic.
This completes the bronze → silver → gold chain end to end.


In [10]:
from dataforge_ai import hourly_demand

GOLD_PATH = f"{DELTA_ROOT}/gold/hourly_demand"

silver = spark.read.format("delta").load(SILVER_PATH)

gold_hourly_demand = hourly_demand(silver)

(
    gold_hourly_demand.write.format("delta")
    .mode("overwrite")
    .save(GOLD_PATH)
)

gold = spark.read.format("delta").load(GOLD_PATH)
print("Gold rows:", gold.count())
gold.filter(F.col("pickup_borough") == "Manhattan").orderBy("pickup_hour").show(24)

Gold rows: 192
+--------------+-----------+----------+------------------+-----------+----------------+
|pickup_borough|pickup_hour|trip_count|          avg_fare|demand_rank|cumulative_trips|
+--------------+-----------+----------+------------------+-----------+----------------+
|     Manhattan|          0|    217349|15.630597380250201|         18|          217349|
|     Manhattan|          1|    155045|15.527113547679713|         19|          372394|
|     Manhattan|          2|    104952| 15.84988413751048|         21|          477346|
|     Manhattan|          3|     71271|16.858576840510153|         22|          548617|
|     Manhattan|          4|     45275|21.787141910546666|         23|          593892|
|     Manhattan|          5|     42241|22.194626784403788|         24|          636133|
|     Manhattan|          6|    107353|16.557750505342202|         20|          743486|
|     Manhattan|          7|    230110|15.025974447003613|         17|          973596|
|     Manhattan| 

## Challenge 2: `whenNotMatchedInsertAll`

Our earlier `MERGE` only updated matching rows. Here we build a batch with
one row that *does* match an existing trip (gets updated) and one row that
deliberately doesn't match anything (a `PULocationID` that can't exist in
real data) — `.whenNotMatchedInsertAll()` inserts that unmatched row as a
brand-new record, all in the same atomic `MERGE`.


In [11]:
match_keys = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID", "trip_distance",
]

silver = spark.read.format("delta").load(SILVER_PATH)
silver_table = DeltaTable.forPath(spark, SILVER_PATH)
merge_condition = " AND ".join(f"t.{c} = s.{c}" for c in match_keys)

before_count = silver.count()

# Row A: an existing row -> matches -> fare gets bumped again.
matched_row = silver.limit(1)
matched_update = matched_row.withColumn("fare_amount", F.col("fare_amount") + 5.0)

# Row B: same row's data, but with a PULocationID that can't exist in real
# NYC TLC data (valid IDs are 1-265) -- guarantees no match on merge_condition,
# so it's treated as a brand-new trip and inserted instead of updated.
new_row = matched_row.withColumn("PULocationID", F.lit(999999))

batch = matched_update.unionByName(new_row)

(
    silver_table.alias("t")
    .merge(batch.alias("s"), merge_condition)
    .whenMatchedUpdate(set={"fare_amount": "s.fare_amount"})
    .whenNotMatchedInsertAll()
    .execute()
)

after_count = spark.read.format("delta").load(SILVER_PATH).count()
print(f"Silver rows before: {before_count:,}")
print(f"Silver rows after:  {after_count:,}  (+{after_count - before_count} inserted)")

spark.read.format("delta").load(SILVER_PATH).filter(
    F.col("PULocationID") == 999999
).select("PULocationID", "DOLocationID", "fare_amount").show()

Silver rows before: 9,301,798
Silver rows after:  9,301,799  (+1 inserted)
+------------+------------+-----------+
|PULocationID|DOLocationID|fare_amount|
+------------+------------+-----------+
|      999999|           1|      97.02|
+------------+------------+-----------+



## Recap

- **Bronze** (`data/delta/bronze/trips`) and **Silver**
  (`data/delta/silver/trips_enriched`) are now real Delta tables, built by
  reusing `read_trips`/`clean_trips`/`join_zones` from `dataforge_ai` — no
  new business logic duplicated, just a new storage layer.
- Every write created a numbered commit in `_delta_log/`, visible via
  `DESCRIBE HISTORY`.
- `mergeSchema` let us evolve Bronze's schema (`ingestion_date`) without
  rewriting existing data.
- `versionAsOf` let us read Bronze exactly as it was before that append.
- `MERGE` atomically corrected 3 fare amounts in Silver — no full-partition
  rewrite needed.

## Challenge

Two follow-ups to try before we move on:

1. **Gold layer**: build a Gold Delta table from Silver using the
   `hourly_demand` function from notebook 04 (already in the package) —
   this completes the bronze → silver → gold chain end to end.
2. **`whenNotMatchedInsertAll`**: our `MERGE` only used `whenMatchedUpdate`.
   Construct a batch with one row that *doesn't* match any existing trip
   (e.g. change one of the `match_keys` to a value that can't exist) and
   add `.whenNotMatchedInsertAll()` to the merge chain — confirm the new
   row appears in Silver's row count afterward.

## Further reading
- [Delta Lake: Table batch reads and writes](https://docs.delta.io/latest/delta-batch.html)
- [Delta Lake: Table utility commands (`DESCRIBE HISTORY`, `VACUUM`, etc.)](https://docs.delta.io/latest/delta-utility.html)
- [Databricks: Medallion architecture](https://www.databricks.com/glossary/medallion-architecture)
- Note: `VACUUM` (physically deleting old file versions) isn't run here —
  it would break the `versionAsOf` time-travel queries above. In a real
  pipeline you'd only vacuum once you no longer need old versions.
